# BootstrapFewShotWithRandomSearch (BootstrapRS)

Build several bootstrapped demo sets and choose among them using the frozen validation split.

**Use it when:** BootstrapFewShot is promising and you can afford multiple candidate programs to reduce dependence on one demo sample.

**What compilation changes:** Demonstrations only, but candidate selection adds a validation-driven search loop.

Important configuration in this benchmark:

- 8 candidate programs in the full profile (2 in smoke)
- two bootstrapped plus two labeled demos per candidate
- single evaluation thread by default

Every notebook loads the canonical 300-row expanded dataset and frozen,
pair-grouped membership: 160 training, 60 validation, and 80 locked-test rows.
A semantic human/AI pair can never cross partitions. Optimizer choices use
validation only; the locked test is released once after the program is frozen.
These scores teach optimizer tradeoffs, not a general AI-detector leaderboard.

In [1]:
import os
import sys
from pathlib import Path

cwd = Path.cwd().resolve()
REPO_ROOT = cwd if (cwd / "chapter06").is_dir() else cwd.parent
if not (REPO_ROOT / "chapter06" / "results" / "expanded_notebooks" / "comparison.json").exists():
    raise RuntimeError("Run this notebook from the repository or chapter06 directory.")
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from chapter06.notebook_support import artifact_paths, learned_program_preview, verify_prompt_artifact
from chapter06.optimizer_runtime import (
    format_result,
    load_frozen_examples,
    published_result,
    run_optimizer,
    split_summary,
)

OPTIMIZER = 'bootstrap-random-search'
splits = load_frozen_examples()
RUN_LIVE = os.getenv("CHAPTER06_RUN_LIVE", "0") == "1"
print(f"optimizer={OPTIMIZER!r}; live={RUN_LIVE}")
print(split_summary(splits))

optimizer='bootstrap-random-search'; live=False
train=160 (human=80, AI=80); validation=60 (human=30, AI=30); test=80 (human=40, AI=40)


## Compile shape

This is the essential DSPy call used by the shared executable runner:

```python
optimizer = dspy.BootstrapFewShotWithRandomSearch(
    metric=exact_match, num_candidate_programs=profile.bootstrap_candidates,
    max_bootstrapped_demos=2, max_labeled_demos=2, num_threads=1,
)
optimized_detector = optimizer.compile(detector, trainset=trainset, valset=valset)
```

`compile` returns a program. The shared runner then evaluates that program on the
untouched 80-row locked test split. The baseline has its own notebook; all other
notebooks report validation and locked-test accuracy separately.

In [2]:
if RUN_LIVE:
    live_run = run_optimizer(
        OPTIMIZER,
        splits=splits,
    )
    result = live_run.summary()
else:
    result = published_result(OPTIMIZER)

print(format_result(result))
print()
print(artifact_paths(OPTIMIZER))

optimizer: bootstrap-random-search
task model: openai/gpt-5.6-luna
final test accuracy: 65.0% (52/80)
optimized validation accuracy: 61.7%
same-model baseline: 53.8%
uplift: +11.2 points
optimization cost: $0.8766
optimization time: 1119.1s
mean inference latency: 1.579s
p95 inference latency: 2.663s

Published artifacts:
- program snapshot: chapter06/results/expanded_notebooks/bootstrap-random-search/full/optimized_program.json
- prompt snapshot: chapter06/results/expanded_notebooks/bootstrap-random-search/full/learned_prompt.json
- chapter comparison: chapter06/CHAPTER_RESULTS.md


## Read the result

The extra compile spend buys candidate selection, not a new instruction. Check whether the held-out gain justifies the search relative to plain BootstrapFewShot.

The saved output above uses the checked-in expanded-dataset result, so opening or
rerunning the notebook is free. The paid run first passed a bounded smoke profile,
then froze its full program using training and validation only. Set
`CHAPTER06_RUN_LIVE=1` before launching Jupyter to reproduce that full protocol;
prompt optimizers require an OpenAI key, while weight optimizers also require the
local PyTorch/Transformers stack. The next cell previews the durable program artifact.

In [3]:
print(learned_program_preview(OPTIMIZER))
print()
print("Frozen program/prompt consistency:", verify_prompt_artifact(OPTIMIZER))

Predictor: detect.predict
Learned instruction (59 characters):
Decide whether the supplied passage was generated by an AI.

Demonstrations: 2
1. is_ai=False: These three QuerySets are separate. The first is a base ~django.db.models.query.QuerySet containing all entries that contain a headline starting with "What". The second is a sub…
2. is_ai=False: Spark SQL supports automatically converting an RDD of JavaBeans into a DataFrame. The BeanInfo, obtained using reflection, defines the schema of the table. Currently, Spark SQL…

Frozen program/prompt consistency: {'checked': True, 'predictors': 1, 'prompt_state_equal': True, 'mismatches': []}


## Apply the pattern

Adapt the compile shape above to your own DSPy program, metric, and frozen
train/validation split. Keep the test set untouched until the optimizer returns,
then report final test accuracy as `correct / test examples` so every optimizer is easy
to compare. Use the separate baseline notebook when you also need uplift.

The complete Chapter 6 rerun is summarized in `CHAPTER_RESULTS.md`; machine-readable
scores, prompts, programs, predictions, timing, cost, versions, hashes, failures,
and retries live under `results/expanded_notebooks/`. Weight-model payloads are
generated locally and Git-ignored; their checked-in manifests retain file hashes,
sizes, configuration, prompts, programs, and scores. Credentials and provider
request bodies are intentionally excluded.